<a href="https://colab.research.google.com/github/DeepanshuSharma1607/Credit_Card_Fraud_Detector/blob/main/notebooks/CreditCard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
import kagglehub
kartik2112_fraud_detection_path = kagglehub.dataset_download('kartik2112/fraud-detection')

print('Data source import complete.')

In [ ]:
import os
x_train = pd.read_csv(os.path.join(kartik2112_fraud_detection_path, 'fraudTrain.csv'))
x_test = pd.read_csv(os.path.join(kartik2112_fraud_detection_path, 'fraudTest.csv'))

In [ ]:
x_train.head(3)

In [ ]:
x_train.tail(3)

In [ ]:
x_test.head(3)

In [ ]:
def shape_size(x):
  print("Shape ",x.shape)
  print("Size ",x.size)

In [ ]:
shape_size(x_test)
shape_size(x_train)

In [ ]:
x_train.info()

In [ ]:
x_train.columns

In [ ]:
x_train.describe()

In [ ]:
import matplotlib.pyplot as plt
CLR_LEGIT, CLR_FRAUD = '#4C9BE8', '#E85D4C'

_df = x_train.copy()
_df['trans_date_trans_time'] = pd.to_datetime(_df['trans_date_trans_time'])
_df['hour'] = _df['trans_date_trans_time'].dt.hour
card_avg = _df.groupby('cc_num')['amt'].transform('mean')
_df['amt_vs_avg'] = _df['amt'] / card_avg.replace(0, np.nan)
legit = _df[_df['is_fraud'] == 0]
fraud = _df[_df['is_fraud'] == 1]

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle('Fraud EDA — key patterns', fontsize=15, fontweight='bold')

counts = _df['is_fraud'].value_counts().sort_index()
bars = axes[0,0].bar(['Legit','Fraud'], counts.values,
                     color=[CLR_LEGIT, CLR_FRAUD], width=0.5, edgecolor='white')
for bar, val in zip(bars, counts.values):
    axes[0,0].text(bar.get_x() + bar.get_width()/2,
                   val + 8000, f'{val:,}', ha='center', fontweight='bold')
axes[0,0].set_title(f'Class imbalance  ({counts[1]/counts.sum()*100:.2f}% fraud)')
axes[0,0].set_ylabel('Count')
axes[0,0].set_ylim(0, counts.max() * 1.15)
axes[0,0].spines[['top','right']].set_visible(False)

cat_rate = (_df.groupby('category')['is_fraud'].mean() * 100).sort_values()
HIGH_RISK = {'misc_net','grocery_pos','gas_transport','entertainment',
             'grocery_net','shopping_net','shopping_pos','misc_pos'}
colors = [CLR_FRAUD if c in HIGH_RISK else CLR_LEGIT for c in cat_rate.index]
axes[0,1].barh(cat_rate.index, cat_rate.values, color=colors, edgecolor='white')
for i, val in enumerate(cat_rate.values):
    axes[0,1].text(val + 0.05, i, f'{val:.1f}%', va='center', fontsize=9)
axes[0,1].set_title('Fraud rate by category')
axes[0,1].set_xlabel('Fraud rate (%)')
axes[0,1].set_xlim(0, cat_rate.max() * 1.4)
axes[0,1].spines[['top','right']].set_visible(False)


hourly = _df.groupby('hour')['is_fraud'].mean() * 100
axes[1,0].plot(hourly.index, hourly.values, color=CLR_FRAUD,
               linewidth=2.5, marker='o', markersize=4)
axes[1,0].fill_between(hourly.index, hourly.values, alpha=0.15, color=CLR_FRAUD)
axes[1,0].set_title('Fraud rate by hour of day')
axes[1,0].set_xlabel('Hour (0 = midnight)')
axes[1,0].set_ylabel('Fraud rate (%)')
axes[1,0].set_xticks(range(0, 24, 2))
axes[1,0].spines[['top','right']].set_visible(False)

ratio_bins = np.linspace(0, 10, 60)
for data, label, color in [(legit,'Legit',CLR_LEGIT),(fraud,'Fraud',CLR_FRAUD)]:
    axes[1,1].hist(data['amt_vs_avg'].clip(upper=10), bins=ratio_bins,
                   alpha=0.65, label=label, color=color,
                   density=True, edgecolor='none')
axes[1,1].axvline(1.0, color='black', linewidth=1.2,
                  linestyle='--', label='avg spend')
axes[1,1].set_title('Amount vs cardholder average')
axes[1,1].set_xlabel('amt / avg (clipped at 10x)')
axes[1,1].set_ylabel('Density')
axes[1,1].legend(fontsize=9)
axes[1,1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('section1_eda.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
x_train.duplicated().sum()

In [ ]:
x_train.isnull().sum()

In [ ]:
x_train.columns

In [ ]:
'''

['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category',
       'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip',
       'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time',
       'merch_lat', 'merch_long', 'is_fraud'],
      dtype='object')


['age','cc_num_id','cc_num_freq',
,'merchant', 'category','job','amt','gender','dist','city_pop',,'amt_avg'
,'high_risk_cat','is_fraud']

'''
print(x_train.loc[5, 'trans_date_trans_time'])
print(pd.to_datetime(x_train.loc[5, 'trans_date_trans_time']))
print(x_train['job'].nunique())

In [ ]:
x_train['trans_date_trans_time']=pd.to_datetime(x_train['trans_date_trans_time'])
x_test['trans_date_trans_time']=pd.to_datetime(x_test['trans_date_trans_time'])
x_train['dob']=pd.to_datetime(x_train['dob'])
x_test['dob']=pd.to_datetime(x_test['dob'])
x_train['age']=(x_train['trans_date_trans_time']-x_train['dob']).dt.days//365
x_test['age']=(x_test['trans_date_trans_time']-x_test['dob']).dt.days//365


In [ ]:
# 'hour','day_of_week','month','is_weekend'
x_train['hour']=x_train['trans_date_trans_time'].dt.hour
x_train['day_of_week']=x_train['trans_date_trans_time'].dt.dayofweek
x_train['month']=x_train['trans_date_trans_time'].dt.month
x_train['is_weekend']=x_train['day_of_week'].isin([5,6]).astype(int)
x_test['hour']=x_test['trans_date_trans_time'].dt.hour
x_test['day_of_week']=x_test['trans_date_trans_time'].dt.dayofweek
x_test['month']=x_test['trans_date_trans_time'].dt.month
x_test['is_weekend']=x_test['day_of_week'].isin([5,6]).astype(int)

In [ ]:
from sklearn.preprocessing import OrdinalEncoder,LabelEncoder

enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

x_train[['cc_num']] = enc.fit_transform(x_train[['cc_num']])
x_test[['cc_num']] = enc.transform(x_test[['cc_num']])

cc_freq = x_train['cc_num'].value_counts()
x_train['cc_num_freq'] = x_train['cc_num'].map(cc_freq)
x_test['cc_num_freq']  = x_test['cc_num'].map(cc_freq).fillna(0)

In [ ]:
enc_gen = LabelEncoder()
enc_gen.fit(x_train['gender'].astype(str))


x_train['gender'] = enc_gen.transform(x_train['gender'].astype(str))

mapping = {c: i for i, c in enumerate(enc_gen.classes_)}
x_test['gender'] = x_test['gender'].astype(str).map(mapping).fillna(-1).astype(int)


In [ ]:
cat=x_train['category'].value_counts()
x_train['cat_freq']=x_train['category'].map(cat)
x_test['cat_freq']=x_test['category'].map(cat).fillna(0)

In [ ]:
merc=x_train['merchant'].value_counts()
x_train['merc_freq']=x_train['merchant'].map(merc)
x_test['merc_freq']=x_test['merchant'].map(merc).fillna(0)

In [ ]:
job=x_train['job'].value_counts()
x_train['job']=x_train['job'].map(job)
x_test['job']=x_test['job'].map(job).fillna(0)

In [ ]:
def haversine(lat1, long1, lat2, long2):
    R = 6371

    lat1, long1, lat2, long2 = map(np.radians, [lat1, long1, lat2, long2])

    dlat  = lat2 - lat1
    dlong = long2 - long1

    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlong/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

    return R * c

x_train['dist'] = haversine(
    x_train['lat'],      x_train['long'],
    x_train['merch_lat'],x_train['merch_long']
)
x_test['dist'] = haversine(
    x_test['lat'],      x_test['long'],
    x_test['merch_lat'],x_test['merch_long']
)

In [ ]:
card_avg=x_train.groupby('cc_num')['amt'].mean()
overall_avg = x_train['amt'].mean()

x_train['avg_amt']=x_train['cc_num'].map(card_avg)
x_test['avg_amt']=x_test['cc_num'].map(card_avg).fillna(overall_avg)

x_test['amt_vs_avg']=x_test['amt']/x_test['avg_amt']
x_train['amt_vs_avg']=x_train['amt']/x_train['avg_amt']

x_train.drop(columns=['avg_amt'],inplace=True)
x_test.drop(columns=['avg_amt'],inplace=True)

In [ ]:
x_train['category'].unique()
high_risk=['misc_net', 'grocery_pos','gas_transport','entertainment','grocery_net', 'shopping_net', 'shopping_pos','misc_pos']
x_train['high_risk_cat']=x_train['category'].isin(high_risk).astype(int)
x_test['high_risk_cat']=x_test['category'].isin(high_risk).astype(int)


In [ ]:
x_train['amt']=np.log1p(x_train['amt'])
x_test['amt']=np.log1p(x_test['amt'])

x_train['city_pop']=np.log1p(x_train['city_pop'])
x_test['city_pop']=np.log1p(x_test['city_pop'])

In [ ]:
y_train=x_train['is_fraud']
y_test=x_test['is_fraud']

In [ ]:
'''
Index(['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category',
       'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip',
       'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time',
       'merch_lat', 'merch_long', 'is_fraud', 'age', 'hour', 'day_of_week',
       'month', 'is_weekend', 'cc_num', 'cc_num_freq', 'cat_freq',
       'merc_freq', 'dist', 'amt_vs_avg', 'high_risk_cat'],
      dtype='object')

'''
drop_col=['Unnamed: 0','cc_num','trans_date_trans_time','merchant','category',
          'first', 'last', 'street','is_fraud', 'city', 'state', 'zip',
       'lat', 'long','dob', 'trans_num', 'unix_time',
       'merch_lat', 'merch_long']

x_train.drop(columns=drop_col,inplace=True)
x_test.drop(columns=drop_col,inplace=True)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle('Feature engineering — distributions', fontsize=15, fontweight='bold')


axes[0,0].hist(x_train['amt'], bins=60,
               color=CLR_LEGIT, alpha=0.8, edgecolor='none')
axes[0,0].set_title('amt after log transform')
axes[0,0].set_xlabel('log(1 + amt)')
axes[0,0].set_ylabel('Count')
axes[0,0].spines[['top','right']].set_visible(False)

axes[0,1].hist(x_train['city_pop'], bins=60,
               color=CLR_FRAUD, alpha=0.8, edgecolor='none')
axes[0,1].set_title('city_pop after log transform')
axes[0,1].set_xlabel('log(1 + city_pop)')
axes[0,1].set_ylabel('Count')
axes[0,1].spines[['top','right']].set_visible(False)

axes[1,0].hist(x_train['cc_num_freq'], bins=50,
               color='#7F77DD', alpha=0.8, edgecolor='none')
axes[1,0].set_title('Card usage frequency (cc_num_freq)')
axes[1,0].set_xlabel('Transactions per card')
axes[1,0].set_ylabel('Number of cards')
axes[1,0].spines[['top','right']].set_visible(False)

feat_cols = ['amt','gender','city_pop','job','age','hour','day_of_week',
             'month','is_weekend','cc_num_freq','cat_freq',
             'merc_freq','dist','amt_vs_avg','high_risk_cat']
corr = x_train[feat_cols].corrwith(y_train).sort_values()
colors_corr = [CLR_FRAUD if v > 0 else CLR_LEGIT for v in corr.values]
axes[1,1].barh(corr.index, corr.values, color=colors_corr, edgecolor='white')
axes[1,1].axvline(0, color='black', linewidth=0.8)
axes[1,1].set_title('Feature correlation with is_fraud')
axes[1,1].set_xlabel('Pearson correlation')
axes[1,1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('section2_features.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
'''
Index(['amt', 'gender', 'city_pop', 'job', 'is_fraud', 'age', 'hour',
       'day_of_week', 'month', 'is_weekend', 'cc_num_id', 'cc_num_freq',
       'cat_freq', 'merc_freq', 'dist', 'amt_vs_avg', 'high_risk_cat'],
      dtype='object')

'''
shape_size(x_train)
shape_size(x_test)

In [ ]:
cols = ['amt','amt_vs_avg','dist','age','cc_num_freq']

for c in cols:
    print("COLUMN:", c)
    print(x_train[c].describe())
    print("Top 5 values:", x_train[c].nlargest(5).values)
    print("Bottom 5 values:", x_train[c].nsmallest(5).values)
    print("-"*40)

In [ ]:
from sklearn.preprocessing import RobustScaler

scalar=RobustScaler()
scalar.fit(x_train)

x_train=scalar.transform(x_train)
x_test=scalar.transform(x_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
reg=LogisticRegression(
    class_weight='balanced',
    max_iter=1000
)
reg.fit(x_train,y_train)

In [ ]:
x_test.columns

In [ ]:
y_pred=reg.predict(x_test)

In [ ]:
from sklearn.metrics import classification_report,confusion_matrix

print(classification_report(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))

In [ ]:
import gc
gc.collect()

In [ ]:
import psutil
ram = psutil.virtual_memory()
print(f'Used: {ram.used/1e9:.1f} GB')
print(f'Available: {ram.available/1e9:.1f} GB')

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(
    n_estimators=35,
    max_depth=10,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

model_rf.fit(x_train, y_train)
y_pred_rf = model_rf.predict(x_test)

print(classification_report(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))

In [ ]:

from xgboost import XGBClassifier
model_xgb = XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    scale_pos_weight=99,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=0.1,
    random_state=42,
    n_jobs=-1
)

model_xgb.fit(x_train, y_train)
y_pred_xgb = model_xgb.predict(x_test)
print(classification_report(y_test, y_pred_xgb))
print(confusion_matrix(y_test, y_pred_xgb))

In [ ]:
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             confusion_matrix, roc_curve, auc)

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle('Model comparison', fontsize=15, fontweight='bold')

models = ['Logistic\nRegression', 'Random\nForest', 'XGBoost']
preds  = [y_pred, y_pred_rf, y_pred_xgb]

f1s   = [f1_score(y_test, p) for p in preds]
precs = [precision_score(y_test, p, zero_division=0) for p in preds]
recs  = [recall_score(y_test, p) for p in preds]

x = np.arange(3)
w = 0.25

axes[0,0].bar(x - w, f1s, width=w, label='F1', color='#7F77DD', edgecolor='white')
axes[0,0].bar(x, precs, width=w, label='Precision', color=CLR_LEGIT, edgecolor='white')
axes[0,0].bar(x + w, recs, width=w, label='Recall', color=CLR_FRAUD, edgecolor='white')

axes[0,0].set_xticks(x)
axes[0,0].set_xticklabels(models)
axes[0,0].set_ylim(0, 1.15)
axes[0,0].set_ylabel('Score')
axes[0,0].set_title('F1 / Precision / Recall')
axes[0,0].legend(fontsize=9)
axes[0,0].spines[['top','right']].set_visible(False)

for i, f in enumerate(f1s):
    axes[0,0].text(i - w, f + 0.02, f'{f:.2f}',
                   ha='center', fontsize=9, fontweight='bold')


cm = confusion_matrix(y_test, y_pred_xgb)

axes[0,1].imshow(cm, cmap='Blues')
axes[0,1].set_xticks([0,1])
axes[0,1].set_yticks([0,1])
axes[0,1].set_xticklabels(['Pred Legit', 'Pred Fraud'])
axes[0,1].set_yticklabels(['Actual Legit', 'Actual Fraud'])
axes[0,1].set_title('XGBoost confusion matrix')

for i in range(2):
    for j in range(2):
        axes[0,1].text(j, i, f'{cm[i,j]:,}',
                       ha='center', va='center',
                       fontsize=12, fontweight='bold',
                       color='white' if cm[i,j] > cm.max()/2 else 'black')


proba_lr  = reg.predict_proba(x_test)[:,1]
proba_rf  = model_rf.predict_proba(x_test)[:,1]
proba_xgb = model_xgb.predict_proba(x_test)[:,1]

fpr_lr, tpr_lr, _ = roc_curve(y_test, proba_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, proba_rf)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, proba_xgb)

auc_lr  = auc(fpr_lr, tpr_lr)
auc_rf  = auc(fpr_rf, tpr_rf)
auc_xgb = auc(fpr_xgb, tpr_xgb)

axes[1,0].plot(fpr_lr, tpr_lr, label=f'LR (AUC={auc_lr:.4f})', linewidth=2, color='#888888')
axes[1,0].plot(fpr_rf, tpr_rf, label=f'RF (AUC={auc_rf:.4f})', linewidth=2, color=CLR_LEGIT)
axes[1,0].plot(fpr_xgb, tpr_xgb, label=f'XGB (AUC={auc_xgb:.4f})', linewidth=2, color=CLR_FRAUD)

axes[1,0].plot([0,1],[0,1], '--', color='gray', linewidth=1)

axes[1,0].set_xlabel('False positive rate')
axes[1,0].set_ylabel('True positive rate')
axes[1,0].set_title('ROC curve')
axes[1,0].legend(fontsize=9)
axes[1,0].spines[['top','right']].set_visible(False)


thresholds = np.linspace(0.1, 0.9, 40)

f1_scores = [
    f1_score(y_test, (proba_xgb >= t).astype(int))
    for t in thresholds
]

best_t = thresholds[np.argmax(f1_scores)]

axes[1,1].plot(thresholds, f1_scores, color=CLR_FRAUD, linewidth=2)

axes[1,1].axvline(best_t, color='black', linewidth=1.2,
                  linestyle='--', label=f'Best = {best_t:.2f}')

axes[1,1].axvline(0.5, color='gray', linewidth=1,
                  linestyle=':', label='Default = 0.50')

axes[1,1].set_xlabel('Decision threshold')
axes[1,1].set_ylabel('F1 score')
axes[1,1].set_title('XGBoost threshold tuning')
axes[1,1].legend(fontsize=9)
axes[1,1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('section3_models.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Best threshold: {best_t:.2f}  →  F1 = {max(f1_scores):.3f}')

In [ ]:
import pickle

with open('model.pkl','wb') as f:
  pickle.dump(model_xgb,f)
